In [1]:
# Cell 1 - Setup
# If needed, uncomment:
# !pip install -q pandas numpy transformers peft torch accelerate

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from peft import LoraConfig, PeftModel, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

SEED = 13
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [2]:
# Cell 2 - Configuration
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
TRAIN_FILE = Path("Train.csv")
EVAL_FILE = Path("TestData.xlsx.csv")
TOP_K = 3
# Set to 5 for smoke-check mode, or None for full evaluation
N_EVAL_LIMIT = None
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_FILE = OUTPUT_DIR / "lora_predictions.jsonl"
ADJUDICATION_QUEUE_FILE = OUTPUT_DIR / "lora_adjudication_queue.csv"
ADJUDICATION_RESOLVED_FILE = OUTPUT_DIR / "lora_adjudication_resolved.csv"
METRICS_FILE = OUTPUT_DIR / "lora_metrics.json"
TRAINED_ADAPTER_DIR = OUTPUT_DIR / "lora_adapter_from_traincsv"
GEN_CONFIG = {
    "do_sample": False,
    "temperature": 0.0,
    "max_new_tokens": 256,
}
TRAIN_CONFIG = {
    "num_train_epochs": 2,
    "learning_rate": 2e-4,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "max_length": 512,
    "logging_steps": 10,
}


In [5]:
# Cell 3 - Load evaluation data (robust CSV parsing)
import csv

rows = []
with open(EVAL_FILE, "r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f)
    header = next(reader, None)

    for row in reader:
        if not row:
            continue
        if len(row) < 3:
            continue

        rows.append({
            "Rank": row[0].strip(),
            "Film": row[1].strip(),
            "Description": ",".join(row[2:]).strip(),
        })

eval_df = pd.DataFrame(rows)

required_cols = {"Rank", "Film", "Description"}
missing = required_cols - set(eval_df.columns)
if missing:
    raise ValueError(f"Missing required columns: {sorted(missing)}")

eval_df["example_id"] = [f"eval_{i:04d}" for i in range(1, len(eval_df) + 1)]
eval_df["prompt"] = eval_df["Description"].astype(str).str.strip()
eval_df["gold_title"] = eval_df["Film"].astype(str).str.strip()

eval_df = eval_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

if N_EVAL_LIMIT is not None:
    eval_df = eval_df.head(N_EVAL_LIMIT).copy()

eval_df = eval_df[["example_id", "Rank", "prompt", "gold_title"]]
display(eval_df.head(3))
print(f"Loaded eval rows: {len(eval_df)}")


,example_id,Rank,prompt,gold_title
0,eval_0084,84,A crew embarks on a mission with an onboard sy...,2001: A Space Odyssey
1,eval_0054,54,A humble working-class man is mistaken for som...,The Great Dictator
2,eval_0071,71,A longtime favorite feels threatened when some...,Toy Story


Loaded eval rows: 100


In [6]:
# Cell 4 - Build fixed catalog
def normalize_title(text: str) -> str:
    text = str(text).strip().lower()
    text = text.replace("’", "'").replace("`", "'")
    text = re.sub(r"\s+", " ", text)
    text = re.sub(r"\s*([:,!?.;])\s*", r"\1", text)
    return text

catalog_titles = sorted(eval_df["gold_title"].dropna().unique().tolist())
catalog_normalized_to_canonical = {normalize_title(t): t for t in catalog_titles}

print(f"Catalog size: {len(catalog_titles)}")
print("Sample titles:", catalog_titles[:5])


Catalog size: 100
Sample titles: ['12 Angry Men', '12th Fail', '2001: A Space Odyssey', 'A Clockwork Orange', 'Alien']


In [20]:
# Cell 5 - Build training data from Train.csv (single-title supervision)
import csv

train_rows = []
last_err = None
for enc in ["utf-8-sig", "utf-8", "cp1252", "latin-1"]:
    try:
        train_rows = []
        with open(TRAIN_FILE, "r", encoding=enc, newline="") as f:
            reader = csv.reader(f)
            header = next(reader, None)

            for row in reader:
                if not row:
                    continue
                if len(row) < 3:
                    continue

                train_rows.append(
                    {
                        "Rank": row[0].strip(),
                        "Film": row[1].strip(),
                        "Description": ",".join(row[2:]).strip(),
                    }
                )
        print(f"Loaded training data with encoding: {enc}")
        break
    except UnicodeDecodeError as e:
        last_err = e
        continue
else:
    raise ValueError(f"Could not decode training file {TRAIN_FILE}. Last error: {last_err}")

train_df = pd.DataFrame(train_rows)
required_cols = {"Rank", "Film", "Description"}
missing_train = required_cols - set(train_df.columns)
if missing_train:
    raise ValueError(f"Missing required columns in training data: {sorted(missing_train)}")

train_df["prompt"] = train_df["Description"].astype(str).str.strip()
train_df["gold_title"] = train_df["Film"].astype(str).str.strip()
train_df = train_df[(train_df["prompt"] != "") & (train_df["gold_title"] != "")].copy()

print(f"Loaded training rows: {len(train_df)}")
display(train_df.head(3))


Loaded training data with encoding: cp1252
Loaded training rows: 150


,Rank,Film,Description,prompt,gold_title
0,101,Memento,A man with memory issues uses an unconventiona...,A man with memory issues uses an unconventiona...,Memento
1,102,Raiders of the Lost Ark,A professor travels abroad for work and races ...,A professor travels abroad for work and races ...,Raiders of the Lost Ark
2,103,Star Wars: Return of the Jedi,Friends stage an intervention for a family mem...,Friends stage an intervention for a family mem...,Star Wars: Return of the Jedi


In [22]:
# Cell 6 - Train LoRA adapter, load finetuned model, and define prompt template
catalog_block = "\n".join([f"- {t}" for t in catalog_titles])

SYSTEM_INSTRUCTION = (
    "You are a movie recommendation ranking engine. "
    "Return only strict JSON and no other text."
)

def build_user_prompt(description: str) -> str:
    return (
        "Given the movie description below, choose exactly 3 ranked movie titles from the allowed catalog.\n\n"
        "Requirements:\n"
        "1) Output exactly this JSON schema: {\"ranked_titles\":[\"title1\",\"title2\",\"title3\"]}\n"
        "2) Exactly 3 unique titles.\n"
        "3) Titles must come only from the allowed catalog.\n"
        "4) Rank best match first.\n"
        "5) No explanation, no markdown, no extra keys.\n\n"
        f"Description:\n{description}\n\n"
        f"Allowed catalog titles:\n{catalog_block}"
    )

# Replace TRAIN_SYSTEM / build_train_user_prompt / build_train_assistant with this:

TRAIN_SYSTEM = (
    "You are a movie recommendation ranking engine. "
    "Return only strict JSON and no other text."
)

def build_train_user_prompt(description: str) -> str:
    return (
        "Given the movie description below, choose exactly 3 ranked movie titles from the allowed catalog.\n\n"
        "Requirements:\n"
        "1) Output exactly this JSON schema: {\"ranked_titles\":[\"title1\",\"title2\",\"title3\"]}\n"
        "2) Exactly 3 unique titles.\n"
        "3) Titles must come only from the allowed catalog.\n"
        "4) Rank best match first.\n"
        "5) No explanation, no markdown, no extra keys.\n\n"
        f"Description:\n{description}\n\n"
        f"Allowed catalog titles:\n{catalog_block}"
    )

def build_train_assistant(title: str) -> str:
    # Build a valid 3-title target using gold first + deterministic catalog fillers
    fillers = [t for t in catalog_titles if t != title][:2]
    ranked = [title] + fillers
    return json.dumps({"ranked_titles": ranked}, ensure_ascii=False)


tokenizer_train = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer_train.pad_token is None:
    tokenizer_train.pad_token = tokenizer_train.eos_token

base_model_train = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)

lora_cfg = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model_train = get_peft_model(base_model_train, lora_cfg)

train_texts = []
for _, row in train_df.iterrows():
    messages = [
        {"role": "system", "content": TRAIN_SYSTEM},
        {"role": "user", "content": build_train_user_prompt(row["prompt"])},
        {"role": "assistant", "content": build_train_assistant(row["gold_title"])},
    ]
    text = tokenizer_train.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    train_texts.append(text)

encodings = tokenizer_train(
    train_texts,
    truncation=True,
    max_length=TRAIN_CONFIG["max_length"],
    padding="max_length",
)

class PackedDataset(torch.utils.data.Dataset):
    def __init__(self, enc):
        self.enc = enc

    def __len__(self):
        return len(self.enc["input_ids"])

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.enc.items()}
        item["labels"] = item["input_ids"].clone()
        return item

train_dataset = PackedDataset(encodings)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR / "train_tmp"),
    num_train_epochs=TRAIN_CONFIG["num_train_epochs"],
    learning_rate=TRAIN_CONFIG["learning_rate"],
    per_device_train_batch_size=TRAIN_CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=TRAIN_CONFIG["gradient_accumulation_steps"],
    logging_steps=TRAIN_CONFIG["logging_steps"],
)

trainer = Trainer(
    model=model_train,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=DataCollatorForLanguageModeling(tokenizer_train, mlm=False),
)

trainer.train()

TRAINED_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model_train.save_pretrained(TRAINED_ADAPTER_DIR)

print(f"Saved trained adapter to: {TRAINED_ADAPTER_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model_eval = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
model = PeftModel.from_pretrained(base_model_eval, str(TRAINED_ADAPTER_DIR))
model.eval()

print(f"Base model loaded: {MODEL_ID}")
print(f"Adapter loaded: {TRAINED_ADAPTER_DIR}")
print(f"Device: {model.device}")

def generate_raw_response(description: str) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_INSTRUCTION},
        {"role": "user", "content": build_user_prompt(description)},
    ]
    chat_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer([chat_text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        output_ids = model.generate(**model_inputs, **GEN_CONFIG)
    new_tokens = output_ids[0][len(model_inputs.input_ids[0]):]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step,Training Loss
10,1.556384
20,1.157056
30,0.850942


Saved trained adapter to: outputs/lora_adapter_from_traincsv


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Base model loaded: Qwen/Qwen2.5-1.5B-Instruct
Adapter loaded: outputs/lora_adapter_from_traincsv
Device: cuda:0


In [23]:
# Cell 7 - Inference loop (evaluation run)
records = []
for i, row in eval_df.iterrows():
    raw_text = generate_raw_response(row["prompt"])
    records.append(
        {
            "example_id": row["example_id"],
            "gold_title": row["gold_title"],
            "rank": row["Rank"],
            "prompt": row["prompt"],
            "raw_text": raw_text,
        }
    )
    if i < 3:
        print(f"Sample raw output {i + 1}: {raw_text}")
    if (i + 1) % 10 == 0:
        print(f"Generated {i + 1}/{len(eval_df)}")
pred_df = pd.DataFrame(records)
display(pred_df.head(2))
print(f"Inference complete. Rows: {len(pred_df)}")


Sample raw output 1: {"ranked_titles":["The Shawshank Redemption","The Godfather","The Dark Knight"]}
Sample raw output 2: {"ranked_titles":["American Beauty","Fight Club","Inception"]}
Sample raw output 3: {"ranked_titles":["The Shawshank Redemption","The Godfather","The Dark Knight"]}
Generated 10/100
Generated 20/100
Generated 30/100
Generated 40/100
Generated 50/100
Generated 60/100
Generated 70/100
Generated 80/100
Generated 90/100
Generated 100/100


,example_id,gold_title,rank,prompt,raw_text
0,eval_0084,2001: A Space Odyssey,84,A crew embarks on a mission with an onboard sy...,"{""ranked_titles"":[""The Shawshank Redemption"",""..."
1,eval_0054,The Great Dictator,54,A humble working-class man is mistaken for som...,"{""ranked_titles"":[""American Beauty"",""Fight Clu..."


Inference complete. Rows: 100


In [24]:
# Cell 8 - Parse + validate predictions
def parse_and_validate(raw_text: str):
    text = raw_text.strip()
    # 1) direct parse
    try:
        payload = json.loads(text)
    except Exception:
        payload = None
    # 2) fallback: extract first JSON object block
    if payload is None:
        m = re.search(r"\{[\s\S]*\}", text)
        if m:
            try:
                payload = json.loads(m.group(0))
            except Exception:
                payload = None
    if payload is None:
        return None, "invalid_json"
    if not isinstance(payload, dict) or "ranked_titles" not in payload:
        return None, "schema_error"
    ranked = payload["ranked_titles"]
    if not isinstance(ranked, list) or len(ranked) != TOP_K:
        return None, "schema_error"
    if any(not isinstance(x, str) for x in ranked):
        return None, "schema_error"
    normalized = [normalize_title(x) for x in ranked]
    if len(set(normalized)) != TOP_K:
        return None, "duplicate_titles"
    canonical = []
    for n in normalized:
        if n not in catalog_normalized_to_canonical:
            return None, "out_of_catalog"
        canonical.append(catalog_normalized_to_canonical[n])
    return canonical, "ok"
# Parser unit checks
ok_titles = catalog_titles[:3]
synthetic_ok = json.dumps({"ranked_titles": ok_titles})
synthetic_bad_json = "not-json"
synthetic_bad_len = json.dumps({"ranked_titles": ok_titles[:2]})
synthetic_dup = json.dumps({"ranked_titles": [ok_titles[0], ok_titles[0], ok_titles[2]]})
synthetic_ooc = json.dumps({"ranked_titles": ["Fake Movie", ok_titles[1], ok_titles[2]]})
assert parse_and_validate(synthetic_ok)[1] == "ok"
assert parse_and_validate(synthetic_bad_json)[1] == "invalid_json"
assert parse_and_validate(synthetic_bad_len)[1] == "schema_error"
assert parse_and_validate(synthetic_dup)[1] == "duplicate_titles"
assert parse_and_validate(synthetic_ooc)[1] == "out_of_catalog"
print("Parser unit checks passed.")
parsed_titles = []
parse_status = []
for raw in pred_df["raw_text"].tolist():
    parsed, status = parse_and_validate(raw)
    parsed_titles.append(parsed)
    parse_status.append(status)
pred_df["parsed_ranked_titles"] = parsed_titles
pred_df["parse_status"] = parse_status
pred_df["proposed_rank1"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[0] if isinstance(x, list) and len(x) > 0 else None)
pred_df["proposed_rank2"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[1] if isinstance(x, list) and len(x) > 1 else None)
pred_df["proposed_rank3"] = pred_df["parsed_ranked_titles"].apply(lambda x: x[2] if isinstance(x, list) and len(x) > 2 else None)
display(pred_df[["example_id", "parse_status", "proposed_rank1", "proposed_rank2", "proposed_rank3"]].head(5))


Parser unit checks passed.


,example_id,parse_status,proposed_rank1,proposed_rank2,proposed_rank3
0,eval_0084,ok,The Shawshank Redemption,The Godfather,The Dark Knight
1,eval_0054,ok,American Beauty,Fight Club,Inception
2,eval_0071,ok,The Shawshank Redemption,The Godfather,The Dark Knight
3,eval_0046,ok,American Beauty,Fight Club,Inception
4,eval_0045,ok,The Shawshank Redemption,The Godfather,The Dark Knight


In [25]:
# Cell 9 - Reciprocal rank scoring
def reciprocal_rank(gold_title: str, ranked_titles):
    if not isinstance(ranked_titles, list):
        return 0.0
    for idx, title in enumerate(ranked_titles, start=1):
        if title == gold_title:
            return 1.0 / idx
    return 0.0

# Scoring unit checks
test_gold = "The Matrix"
assert reciprocal_rank(test_gold, ["The Matrix", "Inception", "Memento"]) == 1.0
assert reciprocal_rank(test_gold, ["Inception", "The Matrix", "Memento"]) == 0.5
assert reciprocal_rank(test_gold, ["Inception", "Memento", "The Matrix"]) == (1.0 / 3.0)
assert reciprocal_rank(test_gold, ["Inception", "Memento", "Interstellar"]) == 0.0
print("Scoring unit checks passed.")

pred_df["reciprocal_rank_pre"] = pred_df.apply(
    lambda r: reciprocal_rank(r["gold_title"], r["parsed_ranked_titles"]) if r["parse_status"] == "ok" else np.nan,
    axis=1,
)

valid_mask = pred_df["parse_status"] == "ok"
mrr_pre_adjudication = float(pred_df.loc[valid_mask, "reciprocal_rank_pre"].mean()) if valid_mask.any() else 0.0
print(f"Valid rows before adjudication: {valid_mask.sum()} / {len(pred_df)}")
print(f"MRR pre-adjudication (valid rows only): {mrr_pre_adjudication:.6f}")


Scoring unit checks passed.
Valid rows before adjudication: 97 / 100
MRR pre-adjudication (valid rows only): 0.049828


In [26]:
# Cell 10 - Adjudication artifacts
queue_df = pred_df.loc[pred_df["parse_status"] != "ok", [
    "example_id",
    "gold_title",
    "raw_text",
    "parse_status",
    "proposed_rank1",
    "proposed_rank2",
    "proposed_rank3",
]].copy()
queue_df["notes"] = ""
queue_df.to_csv(ADJUDICATION_QUEUE_FILE, index=False)
print(f"Adjudication queue rows: {len(queue_df)} -> {ADJUDICATION_QUEUE_FILE}")

pred_df["final_ranked_titles"] = pred_df["parsed_ranked_titles"]

if ADJUDICATION_RESOLVED_FILE.exists():
    resolved = pd.read_csv(ADJUDICATION_RESOLVED_FILE)
    expected_cols = {"example_id", "resolution_status", "rank1_title", "rank2_title", "rank3_title"}
    missing_cols = expected_cols - set(resolved.columns)
    if missing_cols:
        raise ValueError(f"Missing columns in resolved adjudication file: {sorted(missing_cols)}")

    for _, row in resolved.iterrows():
        if str(row["resolution_status"]).strip().lower() != "accepted":
            continue
        candidate = [row["rank1_title"], row["rank2_title"], row["rank3_title"]]
        candidate_norm = [normalize_title(x) for x in candidate]
        if len(set(candidate_norm)) != TOP_K:
            continue
        if not all(n in catalog_normalized_to_canonical for n in candidate_norm):
            continue
        canonical = [catalog_normalized_to_canonical[n] for n in candidate_norm]
        pred_df.loc[pred_df["example_id"] == row["example_id"], "final_ranked_titles"] = [canonical]
else:
    print(f"No resolved adjudication file found at {ADJUDICATION_RESOLVED_FILE}. Using parsed outputs only.")

pred_df["reciprocal_rank_final"] = pred_df.apply(
    lambda r: reciprocal_rank(r["gold_title"], r["final_ranked_titles"]),
    axis=1,
)
mrr_final = float(pred_df["reciprocal_rank_final"].mean()) if len(pred_df) > 0 else 0.0
print(f"MRR final: {mrr_final:.6f}")


Adjudication queue rows: 3 -> outputs/lora_adjudication_queue.csv
No resolved adjudication file found at outputs/lora_adjudication_resolved.csv. Using parsed outputs only.
MRR final: 0.048333


In [27]:
# Cell 11 - Save outputs + summary
pred_records = []
for _, row in pred_df.iterrows():
    pred_records.append(
        {
            "example_id": row["example_id"],
            "gold_title": row["gold_title"],
            "raw_text": row["raw_text"],
            "parsed_ranked_titles": row["parsed_ranked_titles"] if isinstance(row["parsed_ranked_titles"], list) else None,
            "parse_status": row["parse_status"],
            "reciprocal_rank": float(row["reciprocal_rank_final"]),
        }
    )

with PREDICTIONS_FILE.open("w", encoding="utf-8") as f:
    for rec in pred_records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

metrics = {
    "model_id": MODEL_ID,
    "eval_file": str(EVAL_FILE),
    "num_examples": int(len(pred_df)),
    "num_valid": int((pred_df["parse_status"] == "ok").sum()),
    "num_flagged_for_adjudication": int((pred_df["parse_status"] != "ok").sum()),
    "mrr_pre_adjudication": float(mrr_pre_adjudication),
    "mrr_final": float(mrr_final),
}

with METRICS_FILE.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

# End-to-end smoke-check expectations when N_EVAL_LIMIT == 5
if N_EVAL_LIMIT == 5:
    assert PREDICTIONS_FILE.exists(), "Missing predictions output"
    assert ADJUDICATION_QUEUE_FILE.exists(), "Missing adjudication queue output"
    assert METRICS_FILE.exists(), "Missing metrics output"
    print("Smoke-check output assertions passed for 5-row run.")

summary_df = pd.DataFrame([metrics])
display(summary_df)
print(f"Saved predictions: {PREDICTIONS_FILE}")
print(f"Saved adjudication queue: {ADJUDICATION_QUEUE_FILE}")
print(f"Saved metrics: {METRICS_FILE}")


,model_id,eval_file,num_examples,num_valid,num_flagged_for_adjudication,mrr_pre_adjudication,mrr_final
0,Qwen/Qwen2.5-1.5B-Instruct,TestData.xlsx.csv,100,97,3,0.049828,0.048333


Saved predictions: outputs/lora_predictions.jsonl
Saved adjudication queue: outputs/lora_adjudication_queue.csv
Saved metrics: outputs/lora_metrics.json
